# cog20 dense front zoom: does it train, and how do clusters and weights change along the front?

`runs/moo_zoom/dale_all_front27_2seed_50k`: **every network on the cog20 budget front** (27 points,
`f00` = sparsest wiring ... `f26` = densest), retrained with 2 seeds each, plus the unconstrained `control`;
50k steps, batched (all networks of a batch share one trial stream). Point budgets are the exact ones the
front was trained with (`python -m cmc.moo_pick runs/moo/dale_all_nsga3_budget_p48g3_50k --all-front`).

1. **Does it train?** Accuracy per point and task, training loss, whether the rate budget was met.
2. **Task-variance clusters along the front** (Yang 2019), against each cost.
3. **E→E / E→I / I→E / I→I weights along the front.**

Same definitions as `cog20_zoom_analysis.ipynb` (the 4-point zoom).

In [ ]:
import contextlib, io, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from cmc.front import FEASIBLE_MIN_TASK_ACC, compute_front
from cmc.paths import MOO_RUNS_DIR, MOO_ZOOM_RUNS_DIR
from cmc.task import default_config
from cmc.train_cog import load_checkpoint, make_dale_model

ZOOM_DIR = MOO_ZOOM_RUNS_DIR / "dale_all_front27_2seed_50k"
FRONT_DIR = MOO_RUNS_DIR / "dale_all_nsga3_budget_p48g3_50k"
K_RANGE = range(2, 21)
# Points shown individually (task-variance maps, weight histograms): control and a walk along the front.
SHOW = ["control", "f00", "f03", "f08", "f15", "f18", "f26"]

zoom = pd.read_csv(ZOOM_DIR / "runs.csv")
zsum = json.loads((ZOOM_DIR / "summary.json").read_text())
TASKS = zsum["active_tasks"]
ACC = [f"acc_{t}" for t in TASKS]
FRONT_PTS = sorted(p for p in zsum["points"] if p != "control")
POINTS = ["control"] + FRONT_PTS
BUD = pd.DataFrame({p: zsum["points"][p] for p in POINTS}).T[["rate_budget", "conn_budget"]].astype(float)
BUD["lr"], BUD["lc"] = np.log10(BUD.rate_budget), np.log10(BUD.conn_budget)

def load(point, seed):
    with contextlib.redirect_stdout(io.StringIO()):
        model, config, ck = load_checkpoint(ZOOM_DIR / point / f"seed_{seed:02d}.pt")
    return model, ck, model.effective_w_rec().detach().numpy()      # W[pre, post]

def seeds(point):
    return sorted(zoom[zoom.point == point].seed)

def rep_seed(point):
    sub = zoom[zoom.point == point]
    sub = (sub[sub.is_feasible] if sub.is_feasible.any() else sub).sort_values("min_task_acc")
    return int(sub.iloc[len(sub) // 2].seed)

REP = {p: rep_seed(p) for p in POINTS}
N_E = int(load("control", 0)[0].n_e)
print(f"{ZOOM_DIR.name}: {len(zoom)} networks ({len(FRONT_PTS)} front points + control), "
      f"{int(zoom.is_feasible.sum())} feasible, steps={zsum['steps']}")

# 0. Does it train?

## 0.1 Accuracy per point

In [ ]:
agg = zoom.groupby("point").agg(
    n_feasible=("is_feasible", "sum"), min_acc=("min_task_acc", "mean"), min_acc_sd=("min_task_acc", "std"),
    mean_acc=("mean_acc", "mean"), metabolic=("metabolic_cost", "mean"), wiring=("wiring_cost", "mean"),
    conn_frac=("conn_frac", "mean")).reindex(POINTS)
agg = BUD[["rate_budget", "conn_budget"]].join(agg)
display(agg.round(4))

task_acc = zoom.groupby("point")[ACC].mean().reindex(POINTS).set_axis(TASKS, axis=1)
fig, ax = plt.subplots(figsize=(12, 0.32 * len(POINTS) + 1.5))
im = ax.imshow(task_acc.values, cmap="RdYlGn", vmin=0.4, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(TASKS))); ax.set_xticklabels(TASKS, rotation=60, ha="right")
ax.set_yticks(range(len(POINTS)))
ax.set_yticklabels([f"{p}  (r {BUD.rate_budget[p]:.3g}, w {BUD.conn_budget[p]:.3g})" if p != "control" else p
                    for p in POINTS], fontsize=7)
for i in range(len(POINTS)):
    for j in range(len(TASKS)):
        ax.text(j, i, f"{task_acc.values[i, j]:.2f}", ha="center", va="center", fontsize=5.5)
plt.colorbar(im, ax=ax, label="accuracy (mean of 2 seeds)")
ax.set_title("per-task accuracy (rows: front points from sparsest to densest wiring)"); plt.tight_layout(); plt.show()

## 0.2 Training curves and the rate budget

Left: training loss (task loss + cost terms; 500-step running mean), every network, coloured by wiring budget
(control black). Right: training-time rate cost over the last 10% of steps vs the rate budget: on the diagonal =
the budget was met.

In [ ]:
cmap = plt.cm.viridis
norm_c = plt.Normalize(BUD.lc.min(), BUD.lc.max())
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for p in POINTS:
    for s in seeds(p):
        _, ck, _ = load(p, s)
        loss = np.asarray(ck["history"]["loss"])
        sm = np.convolve(loss, np.ones(500) / 500, mode="valid")
        axes[0].plot(np.arange(len(sm)) + 500, sm, lw=0.8 if p != "control" else 2,
                     color="k" if p == "control" else cmap(norm_c(BUD.lc[p])), alpha=0.8)
axes[0].set_yscale("log"); axes[0].set_xlabel("step"); axes[0].set_ylabel("training loss (running mean)")
axes[0].set_title("training loss (black: control)")
plt.colorbar(plt.cm.ScalarMappable(norm=norm_c, cmap=cmap), ax=axes[0], label="log10 wiring budget")
b = zoom[zoom.point != "control"]
sc = axes[1].scatter(b.rate_budget, b.train_rate_cost_tail, c=np.log10(b.conn_budget), cmap=cmap, norm=norm_c, s=25)
lim = [b.rate_budget.min() * 0.8, b.rate_budget.max() * 1.2]
axes[1].plot(lim, lim, "k--", lw=0.8); axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("rate budget"); axes[1].set_ylabel("training rate cost (last 10%)")
axes[1].set_title("was the rate budget met?")
plt.tight_layout(); plt.show()

## 0.3 The points on the front (3D, interactive)

Background: the `cmc.moo` front run (grey x = fails a task, silver = dominated, light blue = front, surface =
front coloured by accuracy, orange plane = 0.6). Coloured: every retrained network here, by wiring budget;
x = infeasible seed; black diamonds = control.

In [ ]:
import plotly.graph_objects as go
OBJ = ("min_task_acc", "metabolic_cost", "wiring_cost")
moo = pd.read_csv(FRONT_DIR / "runs.csv").drop_duplicates(["x0", "x1"]).reset_index(drop=True)
moo["is_feasible"] = moo.min_task_acc >= FEASIBLE_MIN_TASK_ACC
moo["is_pareto"], _ = compute_front(moo.to_dict("records"), OBJ)

def xyz(sub):
    return dict(x=np.log10(sub.metabolic_cost.astype(float)), y=np.log10(sub.wiring_cost.astype(float)),
                z=sub.min_task_acc.astype(float))

def hov(sub):
    return [f"{r.point} seed {r.seed}<br>min_task_acc={r.min_task_acc:.3f}<br>metabolic={r.metabolic_cost:.4f}  "
            f"wiring={r.wiring_cost:.4f}  conn_frac={r.conn_frac:.3f}" for r in sub.itertuples()]

bad, dom, fr = moo[~moo.is_feasible], moo[moo.is_feasible & ~moo.is_pareto], moo[moo.is_pareto]
traces = [
    go.Scatter3d(**xyz(bad), mode="markers", name="moo: fails a task", marker=dict(size=2, color="lightgrey", symbol="x"), hoverinfo="skip"),
    go.Scatter3d(**xyz(dom), mode="markers", name="moo: dominated", marker=dict(size=2, color="silver", opacity=0.5), hoverinfo="skip"),
    go.Scatter3d(**xyz(fr), mode="markers", name=f"moo front ({len(fr)})", marker=dict(size=3, color="#9ecae1"), hoverinfo="skip"),
    go.Mesh3d(**xyz(fr), delaunayaxis="z", intensity=fr.min_task_acc, colorscale="Viridis", cmin=0, cmax=1,
              opacity=0.45, name="front surface", showlegend=True, showscale=False, hoverinfo="skip"),
]
z_ = zoom[zoom.point != "control"]
for feas_, sym, name in ((True, "circle", "retrained, feasible"), (False, "x", "retrained, fails a task")):
    s_ = z_[z_.is_feasible == feas_]
    traces.append(go.Scatter3d(**xyz(s_), mode="markers", name=name, hovertext=hov(s_), hoverinfo="text",
                               marker=dict(size=4, symbol=sym, color=np.log10(s_.conn_budget), colorscale="Viridis",
                                           cmin=BUD.lc.min(), cmax=BUD.lc.max(),
                                           colorbar=dict(title="log10 wiring budget", x=1.02, len=0.6) if feas_ else None)))
c_ = zoom[zoom.point == "control"]
traces.append(go.Scatter3d(**xyz(c_), mode="markers", name="control", hovertext=hov(c_), hoverinfo="text",
                           marker=dict(size=7, color="black", symbol="diamond")))
allx, ally = np.log10(pd.concat([moo.metabolic_cost, zoom.metabolic_cost])), np.log10(pd.concat([moo.wiring_cost, zoom.wiring_cost]))
pad = lambda v: (v.min() - 0.05 * (v.max() - v.min()), v.max() + 0.05 * (v.max() - v.min()))
(x0, x1), (y0, y1) = pad(allx), pad(ally)
traces.append(go.Surface(x=[x0, x1], y=[y0, y1], z=[[FEASIBLE_MIN_TASK_ACC] * 2] * 2, colorscale=[[0, "orange"], [1, "orange"]],
                         opacity=0.2, showscale=False, name="min_task_acc = 0.6", showlegend=True, hoverinfo="skip"))
fig = go.Figure(traces)
fig.update_layout(title="Retrained front networks on the cog20 budget front",
                  scene=dict(xaxis_title="log10(metabolic cost)", yaxis_title="log10(wiring cost)",
                             zaxis_title="min_task_acc", zaxis=dict(range=[0, 1]), aspectmode="cube"),
                  width=980, height=740, legend=dict(x=0.0, y=1.0))
fig.show()

## 0.4 The budget plane

Front run interpolated (colour = min_task_acc), bold line = 0.6. Circles: the retrained points, filled with their
own mean accuracy over 2 seeds (same colour scale), labelled. Control (no budget) in the title.

In [ ]:
x, y = np.log10(moo.rate_budget), np.log10(moo.conn_budget)
fig, ax = plt.subplots(figsize=(9, 7))
tc = ax.tricontourf(x, y, moo.min_task_acc, levels=np.linspace(0, 1, 21), cmap="RdYlGn", alpha=0.8)
ax.tricontour(x, y, moo.min_task_acc, levels=[FEASIBLE_MIN_TASK_ACC], colors="k", linewidths=2.5)
ax.scatter(x[~moo.is_feasible], y[~moo.is_feasible], s=30, marker="x", c="k", alpha=0.5, label="front run: fails a task")
m = zoom[zoom.point != "control"].groupby("point").min_task_acc.mean()
ax.scatter(BUD.lr[m.index], BUD.lc[m.index], c=m.values, cmap="RdYlGn", vmin=0, vmax=1, s=140,
           edgecolors="k", lw=1.2, zorder=5, label="retrained (mean of 2 seeds)")
for p in m.index:
    ax.annotate(p, (BUD.lr[p], BUD.lc[p]), xytext=(5, 5), textcoords="offset points", fontsize=7)
ctrl = zoom[zoom.point == "control"]
ax.set_title(f"min_task_acc over the budget plane (bold: {FEASIBLE_MIN_TASK_ACC}); "
             f"control: {ctrl.min_task_acc.mean():.2f} ± {ctrl.min_task_acc.std():.2f}")
fig.colorbar(tc, ax=ax, label="min_task_acc", ticks=np.linspace(0, 1, 6))
ax.set_xlabel("log10 rate_budget"); ax.set_ylabel("log10 conn_budget"); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

# 1. Task-variance clusters along the front

Same procedure as the 4-point notebook: active neurons (total task variance > 1% of max), each normalised to
its max, k-means k = 2-20 by silhouette; null = each neuron's vector shuffled across tasks (3 shuffles).
Plus, from the stored mean rates: the fraction of neurons active on a task (mean rate > 5% of the
network's max), averaged over tasks.

In [ ]:
def tv_clusters(tv, k_range=K_RANGE, seed=0):
    tv = np.asarray(tv)
    total = tv.sum(axis=0)
    active = total > 1e-2 * total.max()
    norm = tv[:, active] / tv[:, active].max(axis=0, keepdims=True)
    X = norm.T
    scores, labs = {}, {}
    for k in k_range:
        if k >= len(X):
            break
        labs[k] = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)
        scores[k] = silhouette_score(X, labs[k])
    k = max(scores, key=scores.get)
    lab = labs[k]
    labels = np.full(tv.shape[1], -1)
    pref = {c: int(np.argmax(norm[:, lab == c].mean(axis=1))) for c in range(k)}
    order = sorted(range(k), key=lambda c: (pref[c], -norm[pref[c], lab == c].mean()))
    remap = {c: i for i, c in enumerate(order)}
    labels[active] = [remap[c] for c in lab]
    return dict(active=active, X=X, norm_full=tv / np.maximum(tv.max(axis=0, keepdims=True), 1e-30),
                labels=labels, k=k, silhouette=scores[k])

def null_silhouette(X, k, n=3, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        Xs = np.array([rng.permutation(row) for row in X])
        out.append(silhouette_score(Xs, KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Xs)))
    return float(np.mean(out))

CL, rows = {}, []
for p in POINTS:
    for s in seeds(p):
        _, ck, _ = load(p, s)
        cl = tv_clusters(ck["task_variance"]); CL[p, s] = cl
        n_sel = (cl["X"] > 0.5).sum(axis=1)
        mr = np.asarray(ck["mean_rate"])
        r = zoom[(zoom.point == p) & (zoom.seed == s)].iloc[0]
        rows.append({"point": p, "seed": s, "feasible": bool(r.is_feasible), "min_task_acc": r.min_task_acc,
                     "n_active": int(cl["active"].sum()), "k": cl["k"],
                     "silhouette_excess": cl["silhouette"] - null_silhouette(cl["X"], cl["k"]),
                     "tasks_per_neuron": float(n_sel.mean()), "single_task_frac": float((n_sel == 1).mean()),
                     "active_per_task": float((mr > 0.05 * mr.max()).mean())})
tv = pd.DataFrame(rows).join(BUD[["lr", "lc"]], on="point")
display(tv.groupby("point")[["k", "silhouette_excess", "tasks_per_neuron", "single_task_frac", "active_per_task"]]
        .mean().reindex(POINTS).round(3))

## 1.1 Each statistic against each cost

Top row against the wiring budget (colour = rate budget), bottom row against the rate budget (colour = wiring
budget). One dot per network (x = fails a task); dashed line = control mean. Spearman ρ over the feasible
front networks in each title.

In [ ]:
from scipy.stats import spearmanr
STATS = ["k", "tasks_per_neuron", "single_task_frac", "active_per_task", "silhouette_excess", "n_active"]

def against_costs(df, stats, ctrl_vals, ylabel=None):
    fig, axes = plt.subplots(2, len(stats), figsize=(3.6 * len(stats), 7), squeeze=False)
    f = df[df.point != "control"]
    for row, (xc, cc, xl, cl_) in enumerate([("lc", "lr", "log10 wiring budget", "log10 rate"),
                                              ("lr", "lc", "log10 rate budget", "log10 wiring")]):
        for ax, st in zip(axes[row], stats):
            for feas_, mk in ((True, "o"), (False, "x")):
                g = f[f.feasible == feas_]
                sc = ax.scatter(g[xc], g[st], c=g[cc], cmap="plasma", marker=mk, s=22,
                                vmin=f[cc].min(), vmax=f[cc].max())
            ax.axhline(ctrl_vals[st], color="k", ls="--", lw=1)
            ok = f[f.feasible]
            rho = spearmanr(ok[xc], ok[st]).correlation if len(ok) > 3 else np.nan
            ax.set_title(f"{st}  (ρ={rho:+.2f})", fontsize=9); ax.set_xlabel(xl, fontsize=8)
        fig.colorbar(sc, ax=axes[row].tolist(), label=cl_, fraction=0.015, pad=0.01)
    plt.show()

against_costs(tv, STATS, tv[tv.point == "control"][STATS].mean())

## 1.2 Task-variance maps along the front (Yang et al. 2019, Fig. 2b layout)

Points in `SHOW`, representative seed: tasks on y, active units on x grouped by cluster, `cmap="hot"`.

In [ ]:
CLUSTER_COLORS = plt.cm.tab20(np.linspace(0, 1, 20))

def plot_variance(ax, cl, title):
    idx = np.flatnonzero(cl["active"])
    idx = idx[np.lexsort((idx >= N_E, cl["labels"][idx]))]
    im = ax.imshow(cl["norm_full"][:, idx], cmap="hot", aspect="auto", interpolation="nearest", vmin=0, vmax=1)
    ax.set_yticks(range(len(TASKS))); ax.set_yticklabels(TASKS, fontsize=7)
    ax.set_xticks([]); ax.set_title(title, fontsize=8); ax.set_xlabel("Clusters", fontsize=7, labelpad=14)
    labs = cl["labels"][idx]
    y = len(TASKS) - 0.5 + 0.9
    for c in range(cl["k"]):
        pos = np.flatnonzero(labs == c)
        if len(pos):
            ax.plot([pos[0] - 0.5, pos[-1] + 0.5], [y, y], lw=4, solid_capstyle="butt",
                    color=CLUSTER_COLORS[c % 20], clip_on=False)
            ax.text(pos.mean(), y + 1.0, str(c + 1), ha="center", va="top", fontsize=6)
    ax.set_ylim(len(TASKS) - 0.5, -0.5)
    return im

nrows = int(np.ceil(len(SHOW) / 2))
fig, axes = plt.subplots(nrows, 2, figsize=(2 * 4.3, 4.2 * nrows), squeeze=False)
for i, (ax, p) in enumerate(zip(axes.flat, SHOW)):
    cl = CL[p, REP[p]]
    budget = "" if p == "control" else f" r={BUD.rate_budget[p]:.3g} w={BUD.conn_budget[p]:.3g}"
    im = plot_variance(ax, cl, f"{p}{budget} (seed {REP[p]})\n{cl['active'].sum()} active, k={cl['k']}, sil={cl['silhouette']:.2f}")
    if i % 2 == 1:
        ax.set_yticklabels([])
for ax in axes.flat[len(SHOW):]:
    ax.axis("off")
fig.subplots_adjust(hspace=0.45, wspace=0.08)
cb = fig.colorbar(im, ax=axes, ticks=[0, 1], fraction=0.025, pad=0.02)
cb.set_label("Normalized Task Variance", fontsize=7, labelpad=0)
plt.show()

# 2. E→E, E→I, I→E, I→I weights along the front

Per network and block: density (fraction of possible synapses present: E/I-normalised |w| > 1e-2), median
|w| of present synapses, spread (std of log10|w|) and heavy tail (share of total weight in the top 10%).

In [ ]:
BLOCKS = {"E→E": (slice(0, N_E), slice(0, N_E)), "E→I": (slice(0, N_E), slice(N_E, None)),
          "I→E": (slice(N_E, None), slice(0, N_E)), "I→I": (slice(N_E, None), slice(N_E, None))}
BCOL = dict(zip(BLOCKS, ["#d62728", "#ff7f0e", "#1f77b4", "#9467bd"]))

def init_W(ck):
    with contextlib.redirect_stdout(io.StringIO()):
        m = make_dale_model(default_config(n_eachring=ck["config"].get("n_eachring", 16), seed=0), **ck["model_kwargs"])
    return m.effective_w_rec().detach().numpy()

def block_stats(model, W):
    mag_norm = np.abs(W) / model.ei_row_scale().numpy()[:, None]
    mask = model.no_autapse.numpy() > 0
    out = {}
    for b, (pre, post) in BLOCKS.items():
        present = (mag_norm[pre, post] > 1e-2) & mask[pre, post]
        v = np.abs(W[pre, post])[present]
        top = np.sort(v)[::-1]
        out[b] = dict(vals=v, density=present.sum() / mask[pre, post].sum(),
                      median=np.median(v) if len(v) else np.nan,
                      log10_std=np.std(np.log10(v)) if len(v) > 1 else np.nan,
                      top10_share=top[: max(1, len(top) // 10)].sum() / top.sum() if len(v) else np.nan)
    return out

VALS, rows = {}, []
for p in POINTS + ["init"]:
    for s in seeds("control" if p == "init" else p):
        model, ck, W = load("control" if p == "init" else p, s)
        if p == "init":
            W = init_W(ck)
        for b, d in block_stats(model, W).items():
            VALS.setdefault((p, b), []).append(d.pop("vals"))
            rows.append({"point": p, "seed": s, "block": b, **d})
VALS = {k: np.concatenate(v) for k, v in VALS.items()}
wst = pd.DataFrame(rows).join(BUD[["lr", "lc"]], on="point")
wst = wst.merge(tv[["point", "seed", "feasible"]], on=["point", "seed"], how="left")

## 2.1 Block statistics against each cost

Top row: against the wiring budget; bottom row: against the rate budget. Lines connect the per-point means
(sorted along the x-axis); colours = blocks. Dashed = control, dotted = init (same block colour).

In [ ]:
WM = ["density", "median", "log10_std", "top10_share"]
fig, axes = plt.subplots(2, len(WM), figsize=(4.2 * len(WM), 7.5))
f = wst[~wst.point.isin(["control", "init"])]
for row, (xc, xl) in enumerate([("lc", "log10 wiring budget"), ("lr", "log10 rate budget")]):
    for ax, m in zip(axes[row], WM):
        for b in BLOCKS:
            g = f[f.block == b]
            ax.scatter(g[xc], g[m], s=8, color=BCOL[b], alpha=0.35)
            mean = g.groupby("point")[[xc, m]].mean().sort_values(xc)
            ax.plot(mean[xc], mean[m], "-", lw=1.2, color=BCOL[b], label=b)
            ax.axhline(wst[(wst.point == "control") & (wst.block == b)][m].mean(), color=BCOL[b], ls="--", lw=0.8)
            ax.axhline(wst[(wst.point == "init") & (wst.block == b)][m].mean(), color=BCOL[b], ls=":", lw=0.8)
        ax.set_title(m); ax.set_xlabel(xl)
        if m in ("density", "median"):
            ax.set_yscale("log")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()
display(wst.groupby(["point", "block"])[WM].mean().unstack("block").reindex(["init"] + POINTS).round(4))

## 2.2 Distributions for the `SHOW` points (present synapses, pooled over seeds)

In [ ]:
ORDER = ["init"] + SHOW
scol = dict(zip(SHOW, plt.cm.viridis(np.linspace(0, 0.9, len(SHOW)))))
scol["init"], scol["control"] = "#bbbbbb", "black"
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, b in zip(axes.flat, BLOCKS):
    allv = np.concatenate([VALS[(p, b)] for p in ORDER if len(VALS[(p, b)])])
    bins = np.logspace(np.log10(np.percentile(allv, 0.1)), np.log10(allv.max()), 60)
    for p in ORDER:
        v = VALS[(p, b)]
        if len(v):
            dens = wst[(wst.point == p) & (wst.block == b)].density.mean()
            ax.hist(v, bins=bins, density=True, histtype="step", lw=2.2 if p in ("init", "control") else 1.3,
                    color=scol[p], ls="--" if p == "init" else "-", label=f"{p} ({dens:.1%})")
    ax.set_xscale("log"); ax.set_title(b); ax.set_xlabel("|w|"); ax.set_ylabel("density"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

# Readout

*(Fill in.)* Does the clustering track one cost (wiring or rate) or both? Do the weight statistics follow
only the wiring budget, as the 4-point zoom suggested (control ≈ rate_limited, wiring_limited ≈ knee)?